In [4]:
import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

set_tracing_disabled(True)

ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",   # Ollama's OpenAI-compatible endpoint
    api_key="ollama"                        # Dummy key — Ollama doesn't need one
)

local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    )
)

agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model             # <-- This is the key line!
)

result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

In [6]:
from openai import AsyncOpenAI
from agents import (
    Agent, Model, ModelProvider,
    OpenAIChatCompletionsModel, Runner,
    set_tracing_disabled
)

set_tracing_disabled(True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen3.5:cloud"


class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""
    
    def __init__(self, model_name: str = OLLAMA_MODEL):
        self.model_name = model_name
        self.client = AsyncOpenAI(
            base_url=OLLAMA_BASE_URL,
            api_key="ollama"
        )
    
    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name,
            openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model()  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

Ollama is a lightweight tool that lets you run large language models locally on your own computer. It simplifies the process of downloading and managing open-source AI models, enabling private and offline usage without relying on cloud services.


In [8]:
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(True)

agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    # IMPORTANT: prepend 'ollama_chat/' — tells LiteLLM to use Ollama's chat API
    model=LitellmModel(model="ollama_chat/qwen3.5:cloud", base_url="http://localhost:11434")
)

result = await Runner.run(agent, "Hello! What model are you?")
print(result.final_output)

Hello! I'm **Qwen3.5**, the latest large language model in the Qwen series. I have a **256K context window**, support **over 100 languages**, and my knowledge is up to date until **2026**. I'm designed to handle complex tasks like advanced reasoning, code generation, and multi-step problem-solving. How can I assist you today? 😊


In [9]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)

def get_ollama_model(
    model_name: str = "qwen3.5:cloud",
    base_url: str = "http://localhost:11434/v1"
):
    """Create an Ollama-backed model for the Agents SDK.
    
    Args:
        model_name: Ollama model name (must be pulled first)
        base_url: Ollama server URL
    
    Returns:
        OpenAIChatCompletionsModel ready to use with Agent(model=...)
    """
    client = AsyncOpenAI(base_url=base_url, api_key="ollama")
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)
print("ollama_setup module ready")

ollama_setup module ready


In [12]:
from pydantic import BaseModel, Field
from typing import Literal, List
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

# Disable tracing
set_tracing_disabled(True)

# ---- FIX: Proper local model (Ollama via LiteLLM) ----
local_model = LitellmModel(
    model="ollama_chat/qwen3.5:cloud",   # make sure this exists in `ollama list`
    base_url="http://localhost:11434"
)

# ---- Schema ----
class ResumeAnalysis(BaseModel):
    name: str = Field(description="Candidate name")
    years_experience: int = Field(description="Total years of experience")
    top_skills: List[str] = Field(description="Top 3-5 relevant skills")
    fit_score: int = Field(ge=1, le=100, description="Job fit score 1-100")
    fit_level: Literal["strong", "moderate", "weak"] = Field(description="Overall fit")
    summary: str = Field(description="2-sentence summary for hiring manager")

# ---- Agent ----
resume_agent = Agent(
    name="Resume Analyzer",
    instructions="""
You are an HR AI assistant that analyzes resumes against job requirements.

JOB: Senior Python Developer
REQUIREMENTS: 5+ years Python, FastAPI/Django, SQL, cloud experience (AWS/GCP)
NICE TO HAVE: AI/ML, Docker, Kubernetes, system design

IMPORTANT:
- Return ONLY valid JSON
- Use EXACT values:
  fit_level: strong, moderate, weak
- top_skills must be a list of 3-5 skills

Format:
{
  "name": "...",
  "years_experience": number,
  "top_skills": ["...", "..."],
  "fit_score": number,
  "fit_level": "...",
  "summary": "..."
}
""",
    model=local_model,
    output_type=ResumeAnalysis
)

# ---- Input Resume ----
resume_text = """
Ahmed Hassan — Lahore, Pakistan
7 years experience in software development.
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, AWS (EC2, Lambda, S3)
Previous: Lead Developer at SAI Lab (3 years), Senior Dev at SAI (4 years)
Projects: Built a real-time analytics pipeline processing 1M events/day.
Education: BS Computer Science, UAF
"""

# ---- Run ----
result = await Runner.run(resume_agent, resume_text)
r = result.final_output

# ---- Output ----
print(f"{r.name}")
print(f"Experience: {r.years_experience} years")
print(f"Skills: {', '.join(r.top_skills)}")
print(f"Fit Score: {r.fit_score}/100 ({r.fit_level})")
print(f"Summary: {r.summary}")

Ahmed Hassan
Experience: 7 years
Skills: Python, FastAPI, AWS, PostgreSQL, Docker
Fit Score: 95/100 (strong)
Summary: Ahmed exceeds the 5+ year requirement with 7 years of experience. He possesses all core skills including Python, FastAPI, SQL (PostgreSQL), and AWS. Additionally, he has Docker experience and a background in leading development teams and building high-scale analytics pipelines.
